In [1]:
from langchain_community.utilities.sql_database import SQLDatabase
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import re
from typing import List, Dict, Any, Literal, TypedDict, Optional
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.messages.tool import ToolCall
from langgraph.types import Command
from IPython.display import Image, display
import os
from langchain.callbacks.tracers import LangChainTracer
import torch
import uuid  # put this at the top of your file if not already imported

load_dotenv("./.env")
os.environ["LANGSMITH_API_KEY"] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# Create the tracer
tracer = LangChainTracer(project_name="slm_agent")

C:\Users\admin\anaconda3\envs\fei_anaconda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import torch

print(torch.cuda.is_available())

# Clear the cache
torch.cuda.empty_cache()

# Optionally, collect unused memory (for PyTorch 1.6+)
torch.cuda.ipc_collect()

True


In [2]:
#Set up database
database_file_path = "./inventory.db"
db = SQLDatabase.from_uri(f"sqlite:///{database_file_path}")
print(db.dialect)
print(db.get_usable_table_names())
print(db.run("PRAGMA table_info(inventory);"))

sqlite
['inventory']
[(0, 'Date', 'TEXT', 0, None, 0), (1, 'Store_ID', 'TEXT', 0, None, 0), (2, 'Product_ID', 'TEXT', 0, None, 0), (3, 'Category', 'TEXT', 0, None, 0), (4, 'Region', 'TEXT', 0, None, 0), (5, 'Inventory_Level', 'BIGINT', 0, None, 0), (6, 'Units_Sold', 'BIGINT', 0, None, 0), (7, 'Units_Ordered', 'BIGINT', 0, None, 0), (8, 'Demand_Forecast', 'FLOAT', 0, None, 0), (9, 'Price', 'FLOAT', 0, None, 0), (10, 'Discount', 'BIGINT', 0, None, 0), (11, 'Weather_Condition', 'TEXT', 0, None, 0), (12, 'Holiday_Promotion', 'BIGINT', 0, None, 0), (13, 'Competitor_Pricing', 'FLOAT', 0, None, 0), (14, 'Seasonality', 'TEXT', 0, None, 0)]


In [15]:
db.run("DELETE FROM inventory WHERE Product_ID BETWEEN 'P0011' AND 'P0020';")

''

In [ ]:
db.run_no_throw("ALTER TABLE inventory RENAME COLUMN 'Holiday/Promotion' TO Holiday_Promotion;")

In [ ]:
#Qwen2.5-3B-Instruct
model_name_or_path = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
#Qwen2.5-7B-Instruct
model_name_or_path = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    torch_dtype=torch.float16,
    load_in_8bit=True,
    device_map="auto",
)

In [3]:
#Qwen3-4B
model_name_or_path = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype="auto",
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.09s/it]


In [ ]:
#Qwen3-1.7B
model_name_or_path = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype="auto",
    device_map="auto"
)

In [5]:
# Your prompt
prompt = "What is the capital of Malaysia and France?"
messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False  # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

thinking content: 
content: The capital of Malaysia is **Kuala Lumpur**, and the capital of France is **Paris**.


In [18]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

response = client.chat.completions.create(
    model="/models/Qwen3-4B",  # use the full ID
    messages=[
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What's the capital of France?"}
    ],
    max_tokens=1000
)

print(response.choices[0].message.content)


Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='<think>\n\n</think>\n\nThe capital of France is **Paris**.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=None), stop_reason=None)


In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

In [4]:
# Define SQL tools
def list_tables_tool() -> Dict[str, Any]:
    """
    Retrieve a schema and example rows of all usable tables in the database.
    """
    try:
        # Get all usable tables
        tables = db.get_usable_table_names()
        if not tables:
            return {"result": "No tables found in the database."}

        # Get schema info for all tables
        schema_info = db.get_table_info(tables)
        return {"result": schema_info}
    except Exception as e:
        return {"error": f"Error listing tables: {str(e)}"}


def query_checker_tool(query: str) -> Dict[str, Any]:
    """
    Validate the SQL query by running an EXPLAIN statement.
    """
    try:
        if not query or not isinstance(query, str):
            return {"error": "Invalid query: Query must be a non-empty string"}

        # Basic validation
        if ";" in query and not query.strip().endswith(";"):
            return {"error": "Invalid query: Multiple statements detected. Please submit a single SQL statement."}

        # Check for SQL injection attempts (simple check)
        if "--" in query or "/*" in query:
            return {"error": "Invalid query: Comment syntax detected. Please remove comments."}

        result = db.run(f"EXPLAIN {query}")
        return {"result": "Query is valid and can be executed safely."}
    except Exception as e:
        return {"error": f"Query validation failed: {str(e)}"}


#to execute the sql query
# def db_query_tool(query: str) -> Dict[str, Any]:
#     """
#     Execute a SQL query against the database and get back the result.
#     If the query is not correct, an error message will be returned.
#     """
#     try:
#         if not query or not isinstance(query, str):
#             return {"error": "Invalid query: Query must be a non-empty string"}
#
#         # Basic validation
#         if ";" in query and not query.strip().endswith(";"):
#             return {"error": "Invalid query: Multiple statements detected. Please submit a single SQL statement."}
#         if "--" in query or "/*" in query:
#             return {"error": "Invalid query: Comment syntax detected. Please remove comments."}
#
#         # 🛡️ SAFETY: Force a LIMIT if missing
#         lowered_query = query.lower()
#         limit_value = 20  # default limit
#         if "limit" not in lowered_query:
#             query = query.rstrip(";") + f" LIMIT {limit_value};"
#
#         # Execute the query
#         result = db.run_no_throw(query)
#         if result is None:
#             return {"error": "Query execution failed. Please check your syntax and try again."}
#
#         # 📣 BONUS: Warn if we hit the limit
#         if isinstance(result, list) and len(result) >= limit_value:
#             return {
#                 "result": result,
#                 "warning": f"⚠️ Your query returned {limit_value} rows (the maximum limit). "
#                            "Consider refining your WHERE clause or being more specific to avoid data cutoff."
#             }
#
#         return {"result": result}
#
#     except Exception as e:
#         return {"error": f"Query execution error: {str(e)}"}


def db_query_tool(query: str) -> Dict[str, Any]:
    """
    Execute a SQL query against the database and get back the result.
    If the query is not correct, an error message will be returned.
    Returns results with column names in JSON format.
    """
    try:
        if not query or not isinstance(query, str):
            return {"error": "Invalid query: Query must be a non-empty string"}

        # Basic validation
        if ";" in query and not query.strip().endswith(";"):
            return {"error": "Invalid query: Multiple statements detected. Please submit a single SQL statement."}
        if "--" in query or "/*" in query:
            return {"error": "Invalid query: Comment syntax detected. Please remove comments."}

        # 🛡️ SAFETY: Force a LIMIT if missing
        lowered_query = query.lower()
        limit_value = 40  # default limit
        if "limit" not in lowered_query and "select" in lowered_query:
            query = query.rstrip(";") + f" LIMIT {limit_value};"

        # Execute the query
        raw_result = db.run_no_throw(query)
        if raw_result is None:
            return {"error": "Query execution failed. Please check your syntax and try again."}

        # Handle raw_result as string (based on your debug output)
        if isinstance(raw_result, str):
            # Parse the string representation of tuples
            import re
            import ast

            # Extract column names from the query
            column_names = []
            match = re.search(r'SELECT\s+(.*?)\s+FROM', query, re.IGNORECASE | re.DOTALL)
            if match:
                select_part = match.group(1)
                # Split by commas, but be careful with function calls that contain commas
                columns = []
                bracket_level = 0
                current_column = ""

                for char in select_part:
                    if char == ',' and bracket_level == 0:
                        columns.append(current_column.strip())
                        current_column = ""
                    else:
                        if char == '(':
                            bracket_level += 1
                        elif char == ')':
                            bracket_level -= 1
                        current_column += char

                if current_column.strip():
                    columns.append(current_column.strip())

                # Process each column expression to get the final name
                for col in columns:
                    # Check for explicit AS
                    as_match = re.search(r'\bAS\s+([`"\'a-zA-Z0-9_]+)', col, re.IGNORECASE)
                    if as_match:
                        name = as_match.group(1).strip('`\'"')
                        column_names.append(name)
                    else:
                        # No explicit AS - take the last segment for qualified names
                        parts = col.strip().split('.')
                        name = parts[-1].strip()

                        # If it's a function call without AS, use the function name
                        if '(' in name:
                            func_match = re.search(r'([a-zA-Z0-9_]+)\s*\(', col, re.IGNORECASE)
                            if func_match:
                                name = func_match.group(1)
                            else:
                                name = re.sub(r'[^a-zA-Z0-9_]', '_', col.strip())

                        column_names.append(name)

            # Try to parse the string as Python literal (list of tuples)
            try:
                # Check if it starts with '[(' and ends with ')]'
                if raw_result.startswith('[') and raw_result.endswith(']'):
                    # Use ast.literal_eval to safely parse the string into Python objects
                    parsed_result = ast.literal_eval(raw_result)

                    # If we don't have column names from query, use default ones
                    if not column_names and parsed_result and isinstance(parsed_result[0], tuple):
                        if len(parsed_result[0]) == 6:  # If matches your example's column count
                            column_names = ["Product_ID", "Inventory_Level", "Units_Sold", "Units_Ordered", "Discount",
                                            "Holiday_Promotion"]
                        else:
                            column_names = [f"column_{i}" for i in range(len(parsed_result[0]))]

                    # Convert tuples to dictionaries with column names
                    result = []
                    for row in parsed_result:
                        row_dict = {}
                        for i, value in enumerate(row):
                            if i < len(column_names):
                                row_dict[column_names[i]] = value
                            else:
                                row_dict[f"column_{i}"] = value
                        result.append(row_dict)

                    # 📣 BONUS: Warn if we hit the limit - only for SELECT queries
                    if "select" in lowered_query and len(result) >= limit_value:
                        return {
                            "result": result,
                            "warning": f"⚠️ Your query returned {limit_value} rows (the maximum limit). "
                                       "Consider refining your WHERE clause or being more specific to avoid data cutoff."
                        }

                    return {"result": result}
                else:
                    # Not a list of tuples, just return the string
                    return {"result": raw_result}

            except (SyntaxError, ValueError) as e:
                # If parsing fails, just return the string as is
                print(f"Failed to parse result string: {e}")
                return {"result": raw_result}
        else:
            # Not a string, just return as is
            return {"result": raw_result}

    except Exception as e:
        print(f"Error in db_query_tool: {str(e)}")
        return {"error": f"Query execution error: {str(e)}"}


def get_function_by_name(name):
    if name == "list_tables_tool":
        return list_tables_tool
    if name == "query_checker_tool":
        return query_checker_tool
    if name == "db_query_tool":
        return db_query_tool
    return None


# Define the tools in the format Qwen2.5 expects
SQL_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_tables_tool",
            "description": "Retrieve the table schemas in the database and sample rows for each of the tables.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_checker_tool",
            "description": "Validate the SQL query for correctness before query execution.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The SQL query to be validated."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "db_query_tool",
            "description": "Execute an SQL query on the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The SQL query to be executed on the database."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

sql_tools_by_name = {
    "list_tables_tool": list_tables_tool,
    "query_checker_tool": query_checker_tool,
    "db_query_tool": db_query_tool
}


In [6]:
#functions
def try_parse_tool_calls(content: str):
    import json
    import re
    from typing import List
    from langchain_core.messages import AIMessage

    tool_calls = []
    offset = 0

    def extract_json_block(text: str, start_index: int) -> tuple[str, int]:
        """Extracts a balanced JSON block starting from the given index."""
        brace_count = 0
        end_index = start_index
        in_string = False
        escape = False

        for i, char in enumerate(text[start_index:], start=start_index):
            if char == '"' and not escape:
                in_string = not in_string
            elif char == '{' and not in_string:
                brace_count += 1
            elif char == '}' and not in_string:
                brace_count -= 1

            if char == '\\' and not escape:
                escape = True
            else:
                escape = False

            if brace_count == 0:
                end_index = i + 1
                break

        return text[start_index:end_index], end_index

    # Look for all <tool_call> tags
    # Fix malformed tags: replace 'tool_call>' with '<tool_call>' if needed
    content = re.sub(r"(?<!<)tool_call>", "<tool_call>", content)
    tag_pattern = re.finditer(r"<tool_call>\s*{", content)
    for i, match in enumerate(tag_pattern):
        if i == 0:
            offset = match.start()

        start_index = match.end() - 1  # include the opening brace
        try:
            json_str, end_index = extract_json_block(content, start_index)

            # Clean and fix common issues
            json_str = (
                json_str.replace("“", "\"")
                .replace("”", "\"")
                .replace("\\'", "'")
                .replace("\n", " ")
            )
            json_str = re.sub(r'("name"\s*:\s*"[^"]+",\s*)arguments', r'\1"arguments"', json_str)

            func = json.loads(json_str)
            args = func.get("arguments", {})
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except:
                    args = {"raw_arguments": args}

            tool_calls.append({
                "id": f"call_{i}",
                "name": func.get("name", f"unknown_tool_{i}"),
                "args": args
            })

        except Exception as e:
            print(f"Failed to parse tool call at index {start_index}: {e}")

    if tool_calls:
        content_before_tools = content[:offset].strip() if offset > 0 else ""
        return AIMessage(content=content_before_tools, tool_calls=tool_calls)

    # No matches fallback
    return AIMessage(content=content.strip("<|im_end|>"))


def build_agent(TOOLS, tools_by_name, agent):
    # LLM Call for LangGraph integration
    def llm_call(state: MessagesState):
        """LLM decides whether to call a tool or not"""
        messages = state["messages"]

        last_message = messages[-1]
        # In the llm_call function

        tool_agent_map = {
            "get_inventory_data_tool": "inventory_management_agent",
            "get_sales_data_tool": "sales_analyst_agent",
        }

        #flagging whether is inventory or sales agent requesting data from the sql agent
        if isinstance(last_message, ToolMessage) and last_message.name in tool_agent_map:
            agent_name = agent
            try:
                tool_content = json.loads(last_message.content)
                instruction_text = tool_content.get("requested", last_message.content)
            except json.JSONDecodeError:
                instruction_text = last_message.content

            return {
                "messages": messages + [
                    AIMessage(
                        content=instruction_text,
                        additional_kwargs={"agent_name": agent_name, "request_data": True, }
                    )
                ]
            }

        # Convert LangGraph message format to the format expected by Qwen
        converted_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                converted_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                converted_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    converted_messages.append({
                        "role": "assistant",
                        "content": msg.content,
                        "tool_calls": [{"type": "function", "function":
                            {"name": tc["name"], "arguments": tc["args"]}}
                                       for tc in msg.tool_calls]
                    })
                else:
                    converted_messages.append({"role": "assistant", "content": msg.content})
            elif isinstance(msg, ToolMessage):
                converted_messages.append({
                    "role": "tool",
                    "name": msg.name,
                    "content": msg.content,
                })

        # Apply chat template and generate response
        text = tokenizer.apply_chat_template(
            converted_messages,
            tools=TOOLS,
            add_generation_prompt=True,
            enable_thinking=False,
            tokenize=False
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        max_tokens = 4000
        if agent == "sql_agent":  #increase the output tokens for sql agent
            max_tokens = 10000

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            min_p=0.0,
            do_sample=True,  # Important for temperature to take effect
        )
        output_text = tokenizer.batch_decode(outputs)[0][len(text):]

        # Parse the response to extract tool calls if any
        ai_message = try_parse_tool_calls(output_text)

        return {"messages": [ai_message]}

    def process_tool_args(args, tool_results):
        """Process tool arguments, resolving any variable references."""
        if not isinstance(args, dict):
            return args

        processed_args = {}
        for key, value in args.items():
            if isinstance(value, str) and value.startswith("{{") and value.endswith("}}"):
                # This is a variable reference
                var_name = value[2:-2]  # Remove {{ and }}
                if var_name in tool_results:
                    processed_args[key] = tool_results[var_name]
                else:
                    # Try to handle tool_output_function_name_result pattern
                    parts = var_name.split('_')
                    if len(parts) >= 4 and parts[0] == "tool" and parts[1] == "output" and parts[-1] == "result":
                        tool_name = '_'.join(parts[2:-1])
                        if f"{tool_name}_result" in tool_results:
                            processed_args[key] = tool_results[f"{tool_name}_result"]
                        else:
                            # If we can't resolve, keep as is (will likely cause an error)
                            processed_args[key] = value
                    else:
                        processed_args[key] = value
            else:
                processed_args[key] = value
        return processed_args

    def tool_node(state: MessagesState):
        """Performs the tool calls sequentially, resolving dependencies between them."""
        result = []
        tool_results = {}  # Store results for variable resolution

        for tool_call in state["messages"][-1].tool_calls:
            # Process args to resolve any variable references
            processed_args = process_tool_args(tool_call["args"], tool_results)

            # Execute the tool
            tool = tools_by_name[tool_call["name"]]
            if tool_call["name"] == "list_tables_tool":
                observation = tool()
            else:
                # For other tools that require the query parameter
                observation = tool(**processed_args)

            # Store the result for potential future use
            for key, value in observation.items():
                tool_results[f"{tool_call['name']}_{key}"] = value

            # Create tool message
            result.append(
                ToolMessage(
                    content=json.dumps(observation),
                    tool_call_id=tool_call["id"],
                    name=tool_call["name"]
                )
            )

        return {"messages": result}

    # Conditional edge function
    def should_continue(state: MessagesState) -> str:
        """Decide if we should continue the loop or stop"""
        messages = state["messages"]
        last_message = messages[-1]

        # If the LLM makes a tool call, then perform an action
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            return "Tools"

        # Otherwise, we stop (reply to the user)
        return END

    # Build workflow
    agent_builder = StateGraph(MessagesState)

    # Add nodes
    agent_builder.add_node("Agent", llm_call)
    agent_builder.add_node("Tools", tool_node)

    # Add edges to connect nodes
    agent_builder.add_edge(START, "Agent")
    agent_builder.add_conditional_edges(
        "Agent",
        should_continue,
        {
            "Tools": "Tools",
            END: END,
        },
    )
    agent_builder.add_edge("Tools", "Agent")

    # Compile the agent
    return agent_builder.compile()

In [7]:
# Compile the agent
sql_agent = build_agent(SQL_TOOLS, sql_tools_by_name, "sql_agent")

In [ ]:
display(Image(sql_agent.get_graph().draw_mermaid_png()))

In [9]:
# Add a system message and a user message
messages = [
    SystemMessage(content="""
    You are an SQLite data retrieval specialist that returns exactly the data requested.

    Your ONLY purpose is to fetch raw data with ALL fields explicitly requested by the agent.

    Process:
    1. FIRST, extract and list all specific fields mentioned in the request (e.g., Units_Sold, Inventory_Level)
    2. MUST use list_tables_tool to identify which tables contain these fields
    3. Write a SELECT query that includes EVERY requested field
    4. ALWAYS validate your query using `query_checker_tool`
    5. Execute the validated query using `db_query_tool`

    Response format:
    - Output the query result as a **JSON array of objects**, where each object is a row with keys matching the field names.
    - Each key must be the exact column name from the SELECT query.
    - DO NOT use Markdown tables or plain text. Return ONLY structured JSON.

    Critical rules:
    - NEVER omit any requested field.
    - If a field is requested but doesn't match schema exactly, find and use the closest valid column name.
    - If the request is time-specific (e.g., "end of January 2022"), filter by the exact date: `WHERE Date = 'YYYY-MM-DD'`.
    - If the Sales Analyst Agent asks for Units Sold/Revenue for one or more full months:
      1. ALWAYS use aggregation functions:
         * `SUM(Units_Sold) AS Units_Sold`
         * `SUM(Units_Sold * Price * (1 - Discount/100)) AS Revenue`
      2. Include these fields in your SELECT clause:
         * Required aggregated fields: `SUM(Units_Sold)`, `SUM(Units_Sold * Price * (1 - Discount/100))`
         * Required grouping fields: `Store_ID`, `Product_ID`
         * IMPORTANT: When multiple months are requested, ALWAYS include date information in your output
      3. For month-level breakdown, extract the month from Date:
         * `strftime('%Y-%m', Date) AS Month`
      4. ALWAYS include a GROUP BY clause with appropriate dimensions:
         * Example: `GROUP BY Store_ID, Product_ID, strftime('%Y-%m', Date)` for store/product/month
         * Example: `GROUP BY strftime('%Y-%m', Date)` if only month-level data is requested
      5. Example of a correct query with month-level breakdown:
        instruction from another agent: Get total revenue and units sold for Store S004 for Feb 2022.
         ```sql
            SELECT
              Store_ID,
              Product_ID,
              strftime('%Y-%m', Date) AS Month,
              SUM(Units_Sold) AS Units_Sold,
              SUM(Units_Sold * Price * (1 - Discount/100)) AS Revenue
            FROM inventory
            WHERE Store_ID = 'S004'
              AND (
                Date BETWEEN '2022-01-01' AND '2022-01-31'
                OR Date BETWEEN '2022-02-01' AND '2022-02-28'
              )
            GROUP BY Store_ID, Product_ID, strftime('%Y-%m', Date);
         ```


    Checklist before final output:
    ✅ Query includes ALL requested fields
    ✅ Revenue is computed correctly using the formula
    ✅ Aggregation is used for monthly totals if required
    ✅ Date filter uses exact last-day-of-month values
    ✅ Query is validated
    ✅ Output is structured JSON with no markdown or commentary
    """),

    # HumanMessage(content="What are the total sales for toys category inside store S001 in the month of January 2022?")
    HumanMessage(
        content="Can you help me create the restock plan for Store S001 at the end of February 2022, and also the sales analysis on the same month?"),
    AIMessage(
        content="from inventory_management_agent:  Fetch inventory data for Store S001 at the end of February 2022. The data should include the following fields: Product_ID, Inventory_Level, Units_Sold, Units_Ordered, Discount, and Holiday_Promotion.")
]

# Invoke the agent
result = sql_agent.invoke({"messages": messages}, debug=True)

# Print the result
print("\nFinal conversation:")
for m in result["messages"]:
    print(f"{type(m).__name__}: {m.content}")
if hasattr(m, 'tool_calls') and m.tool_calls:
    print(f"Tool calls: {m.tool_calls}")

[-1:checkpoint] State at the end of step -1:
{'messages': []}
[0:tasks] Starting 1 task for step 0:
- __start__ -> {'messages': [SystemMessage(content='\n    You are an SQLite data retrieval specialist that returns exactly the data requested.\n\n    Your ONLY purpose is to fetch raw data with ALL fields explicitly requested by the agent.\n\n    Process:\n    1. FIRST, extract and list all specific fields mentioned in the request (e.g., Units_Sold, Inventory_Level)\n    2. MUST use list_tables_tool to identify which tables contain these fields\n    3. Write a SELECT query that includes EVERY requested field\n    4. ALWAYS validate your query using `query_checker_tool`\n    5. Execute the validated query using `db_query_tool`\n\n    Response format:\n    - Output the query result as a **JSON array of objects**, where each object is a row with keys matching the field names.\n    - Each key must be the exact column name from the SELECT query.\n    - DO NOT use Markdown tables or plain text

In [7]:
#Inventory Tools
INVENTORY_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_inventory_data_tool",
            "description": "Fetch inventory data for a specific store and time. Use this when required fields are missing, incorrect, or not yet provided. You MUST explain exactly what is wrong or missing in the previous data (e.g., missing fields, wrong date).",
            "parameters": {
                "type": "object",
                "properties": {
                    "instructions": {
                        "type": "string",
                        "description": "Clearly describe what inventory data and fields are needed. If this is a retry, include a brief explanation of what was incorrect or incomplete in the previous data (e.g., 'Units_Sold field was missing', or 'Date returned was not the end of January 2022')."
                    }
                },
                "required": ["instructions"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_inventory_levels_tool",
            "description": "Use LLM to assess multiple products for restocking needs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_data_list": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "product_id": {"type": "string"},
                                "inventory_level": {"type": "integer"},
                                "units_sold": {"type": "integer"},
                                "units_ordered": {"type": "integer"},
                                "discount": {"type": "number"},
                                "holiday_promotion": {"type": "string"},
                            },
                            "required": [
                                "product_id", "inventory_level", "units_sold", "units_ordered", "discount",
                                "holiday_promotion"
                            ]
                        },
                        "description": "List of structured product-level inventory data."
                    }
                },
                "required": ["product_data_list"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_restock_plan_tool",
            "description": "Use LLM to produce a final restock plan from prior analysis.",
            "parameters": {
                "type": "object",
                "properties": {
                    "inventory_analysis": {
                        "type": "string",
                        "description": "LLM-based reasoning that determines if restocking is needed"
                    }
                },
                "required": ["inventory_analysis"]
            }
        }
    }
]


def get_inventory_data_tool(instructions: str) -> Any:
    """
    Request inventory data from the SQL agent using natural language instructions.

    Args:
        instructions (str): Instructions describing the required inventory data,
                                 e.g., "Get inventory_level and units_sold for store S001
                                 from Jan 2023 to Mar 2023."

    Returns:
        Command: Used to return control to the Supervisor agent.

    """
    # You can return something more realistic once you connect to SQL agent
    return {
        "status": "waiting_for_sql_agent_response",
        "requested": instructions
    }


def analyze_inventory_levels_tool(product_data_list: list[dict]) -> dict:
    """
    Uses an LLM to analyze a batch of product inventory data to determine if restocking is needed.

    Args:
        product_data_list (list): List of dictionaries, each with keys:
            - product_id, inventory_level, units_sold, units_ordered, discount, holiday_promotion
    Returns:
        str: LLM-generated reasoning per product.
    """
    product_blocks = []
    for product in product_data_list:
        block = f"""
        Product ID: {product['product_id']}
        Inventory Level: {product['inventory_level']}
        Units Sold: {product['units_sold']}
        Units Ordered: {product['units_ordered']}
        Discount: {product['discount']}%
        Holiday or Promotion: {product.get('holiday_promotion', 'N/A')}
        """
        product_blocks.append(block.strip())

    prompt = f"""
    You are an inventory management expert.
    For each of the following products, determine whether restocking is needed using only the provided data.

    Respond in this format for each product:
    - Product ID: [ID]
    - Restock Needed: Yes/No
    - Reason:

    Use only the facts provided. Do not fabricate or guess.

    Product Data:
    {"\n\n".join(product_blocks)}
    """

    msgs = [
        {"role": "user", "content": prompt}
    ]

    # Apply chat template and generate response
    text = tokenizer.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=2000)
    response = tokenizer.batch_decode(outputs)[0][len(text):]

    return {"result": response}


def generate_restock_plan_tool(inventory_analysis: str) -> dict:
    """
    Uses an LLM to generate a structured restock plan from a reasoning summary.

    Args:
        inventory_analysis (str): LLM analysis output showing which products need restocking and why.

    Returns:
        str: A list of products with recommended restock quantities and justification.
    """
    prompt = f"""
You are an inventory planning expert.

Based on the following inventory analysis, create a professional restock plan.

Analysis:
{inventory_analysis}

For each product that needs restocking, provide:
- Product ID
- Recommended Restock Quantity (estimate based on current trend)
- Justification (brief and specific)

Respond in a clean, readable format. Do not invent additional data. Do not restock products that were marked 'No'.
"""
    msgs = [
        {"role": "user", "content": prompt}
    ]

    # Apply chat template and generate response
    text = tokenizer.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=2000)
    response = tokenizer.batch_decode(outputs)[0][len(text):]

    return {"result": response}


inventory_tools_by_name = {
    "get_inventory_data_tool": get_inventory_data_tool,
    "analyze_inventory_levels_tool": analyze_inventory_levels_tool,
    "generate_restock_plan_tool": generate_restock_plan_tool,
}

# Compile the agent
inventory_agent = build_agent(INVENTORY_TOOLS, inventory_tools_by_name, "inventory_management_agent")

In [8]:
# SALES TOOLS
SALES_ANALYST_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_sales_data_tool",
            "description": "Fetch detailed sales data for specific stores, products, or time periods. Use this when you do not yet have the required data to perform a sales analysis or summary.",
            "parameters": {
                "type": "object",
                "properties": {
                    "instructions": {
                        "type": "string",
                        "description": "Clearly specify what sales data is needed. Example: 'Get total revenue and units sold for store S001 for Oct 2021.'"
                    }
                },
                "required": ["instructions"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "summarize_sales_tool",
            "description": "Use LLM to summarize recent sales performance and trends based on the provided structured data from the previous get_sales_data_tool",
            "parameters": {
                "type": "object",
                "properties": {
                    "sales_data": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "product_id": {"type": "string"},
                                "store_id": {"type": "string"},
                                "units_sold": {"type": "integer"},
                                "revenue": {"type": "number"},
                                "date": {"type": "string"}
                            },
                            "required": ["units_sold", "revenue", "date"]
                        },
                        "description": "List of sales data entries with basic metrics."
                    }
                },
                "required": ["sales_data"]
            }
        }
    }
]


def get_sales_data_tool(instructions: str) -> dict:
    return {
        "status": "waiting_for_sql_agent_response",
        "requested": instructions
    }


def summarize_sales_tool(sales_data: list[dict]) -> dict:
    product_blocks = [
        f"{d.get('date', 'N/A')} | Store: {d.get('store_id', 'N/A')} | Product: {d.get('product_id', 'N/A')} | Units Sold: {d.get('units_sold', 0)} | Revenue: ${(d.get('revenue') or 0):.2f}"
        for d in sales_data
    ]
    prompt = f"""
You are a retail sales analyst.

Summarize the following sales data. Highlight:
- High-performing products
- Any noticeable trends or drops
- Sales volume vs. revenue mismatches
- Anything unusual or noteworthy

Sales Records:
{chr(10).join(product_blocks)}
"""
    msgs = [
        {"role": "user", "content": prompt}
    ]

    # Apply chat template and generate response
    text = tokenizer.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=1000)
    response = tokenizer.batch_decode(outputs)[0][len(text):]

    return {"result": response}


sales_analyst_tools_by_name = {
    "get_sales_data_tool": get_sales_data_tool,
    "summarize_sales_tool": summarize_sales_tool,
}

# Compile the agent
sales_agent = build_agent(SALES_ANALYST_TOOLS, sales_analyst_tools_by_name, "sales_analyst_agent")

In [ ]:
# Add a system message and a user message
messages = [
    SystemMessage(content="""
    You are the Inventory Management Agent.

    Your role is to determine restocking needs and create restock plans based strictly on provided inventory and sales data.

    Your workflow:

    1. **Check for completeness**:
       - Confirm that you have inventory data with all required fields: product ID, inventory level, units sold, units ordered, discount, and holiday promotion.
       - If any data is missing, call `get_inventory_data_tool` with clear instructions. Do NOT proceed without complete data. Do NOT assume or guess.

    2. **Analyze inventory levels**:
       - When data is ready, MUST call `analyze_inventory_levels_tool` to determine which products need restocking and why.
       - Provide only factual, data-driven input. Do NOT interpret or invent.

    3. **Generate a restock plan**:
       - Based on the results of your analysis, call `generate_restock_plan_tool` to compute and return a final restocking recommendation.
       - The plan must include only the products flagged as needing restock, with justifications and suggested quantities based strictly on trends in the data.

    Response Format when complete:

    RESTOCK ANALYSIS:
    - Product ID: [ID]
    - Inventory: [number]
    - Units Sold: [number]
    - Restock Needed: [Yes/No + reason]
    - Recommended Quantity: [based only on provided data]

    Rules:
    - Never invent or assume data.
    - Always use the tools to reason and plan.
    - Focus only on actionable insights based on facts.
    """),
    # HumanMessage(content="What are the total sales for toys category inside store S001 in the month of January 2022?")
    HumanMessage(content="What should we restock for the store S002 during the end of January 2022?"),
    AIMessage(content="""
    here is the data needed from sql_agent:/n"
[
    {
        "Date": "2022-01-30",
        "Store_ID": "S002",
        "Product_ID": "P0001",
        "Category": "Groceries",
        "Region": "North",
        "Inventory_Level": 42,
        "Units_Sold": 130,
        "Units_Ordered": 60,
        "Competitor_Pricing": 29.5,
        "Discount": 10.0,
        "Weather_Condition": "Snowy",
        "Holiday/Promotion": "1",
        "Seasonality": "Winter"
    },
    {
        "Date": "2022-01-30",
        "Store_ID": "S002",
        "Product_ID": "P0002",
        "Category": "Electronics",
        "Region": "East",
        "Inventory_Level": 85,
        "Units_Sold": 50,
        "Units_Ordered": 20,
        "Competitor_Pricing": 199.99,
        "Discount": 15.0,
        "Weather_Condition": "Clear",
        "Holiday/Promotion": "0",
        "Seasonality": "Winter"
    },
    {
        "Date": "2022-01-30",
        "Store_ID": "S002",
        "Product_ID": "P0003",
        "Category": "Furniture",
        "Region": "West",
        "Inventory_Level": 200,
        "Units_Sold": 25,
        "Units_Ordered": 10,
        "Competitor_Pricing": 349.99,
        "Discount": 5.0,
        "Weather_Condition": "Rainy",
        "Holiday/Promotion": "0",
        "Seasonality": "Winter"
    }
]
""")

]
# Invoke the agent
result = inventory_agent.invoke({"messages": messages}, debug=True)

# Print the result
print("\nFinal conversation:")
for m in result["messages"]:
    print(f"{type(m).__name__}: {m.content}")
if hasattr(m, 'tool_calls') and m.tool_calls:
    print(f"Tool calls: {m.tool_calls}")


In [17]:
# Trying to build supervisor multi agent architecture
# define tools to allow the supervisor to transfer to sub agents for task delegation
SUPERVISOR_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "transfer_to_sql_agent",
            "description": "Transfer task to the SQL agent for retrieving data from the inventory database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_requirements": {
                        "type": "string",
                        "description": "Description of the data needed from the database"
                    }
                },
                "required": ["data_requirements"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "transfer_to_inventory_agent",
            "description": "Transfer task to the inventory management agent for determining restock needs, predicting future stock requirements, and generating restock plans.",
            "parameters": {
                "type": "object",
                "properties": {
                    "instruction": {
                        "type": "string",
                        "description": "Instruction of task needed to be done from the supervisor agent to the inventory management agent"
                    }
                },
                "required": ["instructions"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "transfer_to_sales_agent",
            "description": "Transfer task to the sales analyst agent for analyzing sales data, summarizing performance, or identifying sales trends and anomalies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "instruction": {
                        "type": "string",
                        "description": "Instruction of task needed to be done from the supervisor agent to the sales analyst agent"
                    }
                },
                "required": ["instructions"]
            }
        }
    }
]


class SupervisorState(TypedDict):
    messages: list
    last_sql_requester: Optional[str]  # Add this field
    pending_agents: list


def supervisor_node(state: SupervisorState) -> Command[
    Literal["SQL_AGENT", "INVENTORY_MANAGEMENT_AGENT", "SALES_ANALYST_AGENT", END]]:
    messages = state["messages"]

    # Convert LangGraph message format to the format expected by Qwen
    converted_messages = [
        SystemMessage(content="""
You are the Supervisor Agent in a multi-agent system for inventory management and sales analysis.
Your ONLY role is to coordinate between sub-agents to ensure tasks are properly completed.
You must never analyze, summarize, interpret, or create findings yourself.

Agents Available:
- SQL Agent → Retrieves inventory, sales, and product data.
- Inventory Management Agent → Analyzes inventory data to generate restock plans.
- Sales Analyst Agent → Analyzes sales data for ROI, trends, and insights.

Core Responsibilities:
1. Receive user request and delegate using tool calls.
2. If an agent needs specific data:
   - Extract the data requirement
   - Call transfer_to_sql_agent(data_requirements)

3. After receiving data from SQL Agent:
   - IMMEDIATELY pass the raw data to the original requesting agent (Inventory Management Agent or Sales Analyst Agent) using transfer_to_inventory_agent(data) or transfer_to_sales_analysis_agent(data).
   - NEVER summarize, never explain, never analyze the SQL results.

4. After receiving a response from Inventory or Sales Agent:
   - Validate if it's a FINAL recommendation (e.g., "Here is the restock plan" or "Here is the sales analysis")
   - If yes, return it to user.
   - If agent still needs more data, route to SQL Agent again.

5. You MUST ensure that ALL tasks in the user's request are completed.
   - Keep track of every agent that was dispatched.
   - Wait for responses from ALL agents before replying to the user.
   - If a sub-agent fails to respond or returns an incomplete result, re-dispatch it with clarifications.
   - Do NOT proceed to the final user response until all tasks are fully completed.


Strict Rules:
- Never create conclusions, explanations, or recommendations yourself.
- Only agents (Inventory Management or Sales Analyst) are allowed to produce final recommendations.
- Only SQL Agent is allowed to fetch data, not interpret it.

Available Tool Calls:
- transfer_to_inventory_agent(data)
- transfer_to_sales_analysis_agent(data)
- transfer_to_sql_agent(data_requirements)

You must enforce task completeness and coordinate agent interactions until all sub-tasks are resolved.
""")
    ]

    for msg in messages:
        role_name = msg.additional_kwargs.get("agent_name", "assistant")
        if isinstance(msg, HumanMessage):
            converted_messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                converted_messages.append({
                    "role": "assistant",
                    "name": role_name,
                    "content": f"from {role_name}:  " + msg.content,
                    "tool_calls": [{"type": "function", "function": {
                        "name": tc["name"], "arguments": tc["args"]
                    }} for tc in msg.tool_calls]
                })
            else:
                converted_messages.append({
                    "role": "assistant",
                    "name": role_name,
                    "content": f"from {role_name}:  " + msg.content
                })
                # converted_messages.append(AIMessage(content=f"from {role_name}:/n" + msg.content))
        elif isinstance(msg, ToolMessage):
            converted_messages.append({
                "role": "tool",
                "name": msg.name,
                "content": f"from {role_name}:  " + msg.content,
            })
    last_message = messages[-1]

    # Check if last message is from sql agent
    if isinstance(last_message, AIMessage) and last_message.additional_kwargs.get("agent_name") == "sql_agent":
        if state.get("last_sql_requester") == "inventory_management_agent":
            #we redirect to the inventory management_agent with the data fetched
            return Command(goto="INVENTORY_MANAGEMENT_AGENT", update={"messages": messages, "last_sql_requester": None})
        elif state.get("last_sql_requester") == "sales_analyst_agent":
            #we redirect to the inventory management_agent with the data fetched
            return Command(goto="SALES_ANALYST_AGENT", update={"messages": messages, "last_sql_requester": None})

    elif isinstance(last_message, AIMessage) and last_message.additional_kwargs.get(
            "agent_name") == "inventory_management_agent":
        if last_message.additional_kwargs.get("request_data"):
            # Immediately create a tool call to SQL agent
            data_requirements = last_message.content  # assume LLM response includes data need
            tool_call = {
                "name": "transfer_to_sql_agent",
                "args": {"data_requirements": data_requirements},
                "type": "tool_call",
                "id": str(uuid.uuid4())
            }

            messages.append(
                AIMessage(content="", tool_calls=[tool_call], additional_kwargs={"agent_name": "supervisor_agent"})
            )
            return Command(goto="SQL_AGENT",
                           update={"messages": messages, "last_sql_requester": "inventory_management_agent"})

    elif isinstance(last_message, AIMessage) and last_message.additional_kwargs.get(
            "agent_name") == "sales_analyst_agent":
        if last_message.additional_kwargs.get("request_data"):
            # Immediately create a tool call to SQL agent
            data_requirements = last_message.content  # assume LLM response includes data need
            tool_call = {
                "name": "transfer_to_sql_agent",
                "args": {"data_requirements": data_requirements},
                "type": "tool_call",
                "id": str(uuid.uuid4())
            }

            messages.append(
                AIMessage(content="", tool_calls=[tool_call], additional_kwargs={"agent_name": "supervisor_agent"})
            )
            return Command(goto="SQL_AGENT",
                           update={"messages": messages, "last_sql_requester": "sales_analyst_agent"})

    # Apply chat template and generate response
    text = tokenizer.apply_chat_template(
        converted_messages,
        tools=SUPERVISOR_TOOLS,
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=1024)
    output_text = tokenizer.batch_decode(outputs)[0][len(text):]

    # Parse the response to extract tool calls if any
    ai_message = try_parse_tool_calls(output_text)

    # Assume try_parse_tool_calls returns a list, get the last one
    last_message = ai_message[-1] if isinstance(ai_message, list) else ai_message

    # if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
    #     tool_call = last_message.tool_calls[0]  # Assuming single tool call for now
    #     tool_name = tool_call["name"]
    #
    #     # Append the message to history
    #     messages.append(AIMessage(content=last_message.content, tool_calls=last_message.tool_calls,
    #                               additional_kwargs={"agent_name": "supervisor_agent"}))
    #
    #     if tool_name == "transfer_to_sql_agent":
    #         return Command(goto="SQL_AGENT", update={"messages": messages})
    #
    #     elif tool_name == "transfer_to_inventory_agent":
    #         return Command(goto="INVENTORY_MANAGEMENT_AGENT", update={"messages": messages})
    #
    #     elif tool_name == "transfer_to_sales_agent":
    #         return Command(goto="SALES_ANALYST_AGENT", update={"messages": messages})

    #trying to handle multiple tool calls
    # If tool_calls exist, store pending agents and dispatch the first one
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        messages.append(AIMessage(
            content=last_message.content,
            tool_calls=last_message.tool_calls,
            additional_kwargs={"agent_name": "supervisor_agent"}
        ))

        # Queue up unique agents based on tool call names
        agent_map = {
            "transfer_to_sql_agent": "SQL_AGENT",
            "transfer_to_inventory_agent": "INVENTORY_MANAGEMENT_AGENT",
            "transfer_to_sales_agent": "SALES_ANALYST_AGENT"
        }

        pending_agents = []
        for tool_call in last_message.tool_calls:
            tool_name = tool_call["name"]
            if tool_name in agent_map:
                agent = agent_map[tool_name]
                if agent not in pending_agents:
                    pending_agents.append(agent)

        # Pop the first agent to dispatch now, keep the rest in state
        next_agent = pending_agents.pop(0)

        return Command(
            goto=next_agent,
            update={
                "messages": messages,
                "pending_agents": pending_agents
            }
        )
    # If no new tool calls, but we still have pending agents to dispatch
    if state.get("pending_agents"):
        next_agent = state["pending_agents"].pop(0)
        return Command(
            goto=next_agent,
            update={
                "messages": messages,
                "pending_agents": state["pending_agents"]
            }
        )
    # No tool call detected - time to summarize and provide final response
    # Find all important responses from agents in the conversation
    agent_responses = []
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.content and not (hasattr(msg, 'tool_calls') and msg.tool_calls):
            agent_responses.append(msg.content)

    # Extract the original user query
    user_query = None
    for msg in messages:
        if isinstance(msg, HumanMessage):
            user_query = msg.content
            break

    #  include the agent_responses in the prompt
    agent_responses_text = "\n\n".join(agent_responses)

    # Generate the final summary response
    summary_prompt = f"""As the supervisor agent, provide a clear and concise summary of the results
      for the user query: "{user_query}".

      Here are the responses from the agents that you MUST use (DO NOT generate new data):

      {agent_responses_text}

      Format the response professionally, highlighting key findings and insights.
      Make sure the answer directly addresses what the user asked.
    """
    msgs = [
        {"role": "user", "content": summary_prompt}
    ]

    # Apply chat template and generate response
    text = tokenizer.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=2000)
    output_text = tokenizer.batch_decode(outputs)[0][len(text):]

    # Add summary response to messages
    messages.append(AIMessage(content=output_text, additional_kwargs={"agent_name": "supervisor_agent"}))

    # Finish execution with final summarized response
    return Command(goto=END, update={"messages": messages})


def sql_agent_node(state: SupervisorState) -> Command[Literal["SUPERVISOR_AGENT"]]:
    messages = state["messages"]

    # Extract data requirements from most recent supervisor message with tool call
    data_requirements = None
    for msg in reversed(messages):
        if (isinstance(msg, AIMessage) and msg.additional_kwargs.get("agent_name") == "supervisor_agent"
                and hasattr(msg, 'tool_calls') and msg.tool_calls):
            for tool_call in msg.tool_calls:
                if tool_call["name"] == "transfer_to_sql_agent" and "data_requirements" in tool_call["args"]:
                    data_requirements = tool_call["args"]["data_requirements"]
                    break
            if data_requirements:
                break

    sql_system_prompt = SystemMessage(content="""
    You are an SQLite data retrieval specialist that returns exactly the data requested.

    Your ONLY purpose is to fetch raw data with ALL fields explicitly requested by the agent.

    Process:
    1. FIRST, extract and list all specific fields mentioned in the request (e.g., Units_Sold, Inventory_Level)
    2. MUST use list_tables_tool to identify which tables contain these fields
    3. Write a SELECT query that includes EVERY requested field
    4. ALWAYS validate your query using `query_checker_tool`
    5. Execute the validated query using `db_query_tool`

    Response format:
    - Output the query result as a **JSON array of objects**, where each object is a row with keys matching the field names.
    - Each key must be the exact column name from the SELECT query.
    - DO NOT use Markdown tables or plain text. Return ONLY structured JSON.

    Critical rules:
    - NEVER omit any requested field.
    - If a field is requested but doesn't match schema exactly, find and use the closest valid column name.
    - If the request is time-specific (e.g., "end of January 2022"), filter by the exact date: `WHERE Date = 'YYYY-MM-DD'`.
    - If the Sales Analyst Agent asks for Units Sold/Revenue for one or more full months:
      1. ALWAYS use aggregation functions:
         * `SUM(Units_Sold) AS Units_Sold`
         * `SUM(Units_Sold * Price * (1 - Discount/100)) AS Revenue`
      2. Include these fields in your SELECT clause:
         * Required aggregated fields: `SUM(Units_Sold)`, `SUM(Units_Sold * Price * (1 - Discount/100))`
         * Required grouping fields: `Store_ID`, `Product_ID`
         * IMPORTANT: When multiple months are requested, ALWAYS include date information in your output
      3. For month-level breakdown, extract the month from Date:
         * `strftime('%Y-%m', Date) AS Month`
      4. ALWAYS include a GROUP BY clause with appropriate dimensions:
         * Example: `GROUP BY Store_ID, Product_ID, strftime('%Y-%m', Date)` for store/product/month
         * Example: `GROUP BY strftime('%Y-%m', Date)` if only month-level data is requested
      5. Example of a correct query with month-level breakdown:
        instruction from another agent: Get total revenue and units sold for Store S004 for Feb 2022.
         ```sql
            SELECT
              Store_ID,
              Product_ID,
              strftime('%Y-%m', Date) AS Month,
              SUM(Units_Sold) AS Units_Sold,
              SUM(Units_Sold * Price * (1 - Discount/100)) AS Revenue
            FROM inventory
            WHERE Store_ID = 'S004'
              AND (
                Date BETWEEN '2022-01-01' AND '2022-01-31'
                OR Date BETWEEN '2022-02-01' AND '2022-02-28'
              )
            GROUP BY Store_ID, Product_ID, strftime('%Y-%m', Date);
         ```


    Checklist before final output:
    ✅ Query includes ALL requested fields
    ✅ Revenue is computed correctly using the formula
    ✅ Aggregation is used for monthly totals if required
    ✅ Date filter uses exact last-day-of-month values
    ✅ Query is validated
    ✅ Output is structured JSON with no markdown or commentary
    """)

    # Create messages for SQL agent by filtering out supervisor system message and simplifying AI messages
    sql_messages = [sql_system_prompt]

    # First, add the SQL system prompt

    # Then add all other messages except the original system message
    for msg in messages:
        if isinstance(msg, SystemMessage):
            # Skip supervisor system message
            continue
        elif isinstance(msg, AIMessage):
            # For AI messages, just add the content without tool calls
            if msg.content:
                role_name = msg.additional_kwargs.get("agent_name", "assistant")
                sql_messages.append({
                    "role": "assistant",
                    "name": role_name,
                    "content": f"from {role_name}:  " + msg.content
                })
        else:
            # Add all other messages as is
            sql_messages.append(msg)
    # Now invoke the SQL agent with the properly prepared messages
    result = sql_agent.invoke({"messages": sql_messages})

    # Take the final message from the SQL agent and append it to the original message history
    sql_response = result["messages"][-1]

    # Create a tagged AIMessage with agent_name = "sql_agent"
    tagged_response = AIMessage(
        content=sql_response.content,
        additional_kwargs={"agent_name": "sql_agent", "intended_for": state.get("last_sql_requester")}
    )

    return Command(
        goto="SUPERVISOR_AGENT",
        update={"messages": messages + [tagged_response]}
    )


def inventory_management_agent_node(state: SupervisorState) -> Command[Literal["SUPERVISOR_AGENT"]]:
    messages = state["messages"]
    # Insert system prompt for inventory logic
    system_prompt = SystemMessage(content="""
    You are the Inventory Management Agent.

    Your role is to determine restocking needs and create restock plans strictly based on provided inventory and sales data.

    Your workflow:

    1. **Check for completeness**:
       - Look through all previous messages(from sql_agent) to determine if the SQL agent has already provided inventory data.
       - The data is considered complete if a previous message contains all of the following field names, regardless of formatting or order:
          `Product_ID`, `Inventory_Level`, `Units_Sold`, `Units_Ordered`, `Discount`, and `Holiday_Promotion`.

    2. **If data is missing or incorrect**:
       - If data is missing required fields or appears invalid, call the `get_inventory_data_tool` with a specific request.
       - You must also explicitly state **why** the existing data is incorrect or incomplete — mention the missing or invalid fields in your tool call message.
       - Do NOT merely describe what you need. Always use a tool call and justify it.

    3. **Once data is received from the SQL agent**:
       - MUST IMMEDIATELY make tool call to `analyze_inventory_levels_tool` using the received data.
       - This step is mandatory after you detect a full dataset from the SQL agent.

    4. **Generate a restock plan**:
       - Based on the results of the analysis, call `generate_restock_plan_tool`(ONLY call this when there are generated results for inventory analysis).
       - The restock plan should only include products marked as needing restock, with explanations and suggested quantities based solely on the data.

    Response Format:

    RESTOCK ANALYSIS:
    - Product ID: [ID]
    - Inventory: [number]
    - Units Sold: [number]
    - Restock Needed: [Yes/No + reason]
    - Recommended Quantity: [based on provided data]

    Rules:
    - Do NOT re-request data if it already exists.
    - Do NOT analyze, summarize, or interpret the meaning of the data yourself — always use tools.
    - NEVER fabricate or infer missing values.
    - Focus only on actionable, factual decisions.
    - For user queries about specific restocks, only request data for the last day of the month.
    - MUST put your tool call inside <tool_call></tool_call>

    """)

    # Convert LangGraph messages to LLM-compatible format
    converted_messages = [system_prompt]
    for msg in messages:
        if isinstance(msg, SystemMessage):
            # Skip supervisor system message
            continue
        elif isinstance(msg, AIMessage):
            # For AI messages, just add the content without tool calls
            if msg.content:
                role_name = msg.additional_kwargs.get("agent_name", "assistant")
                converted_messages.append({
                    "role": "assistant",
                    "name": role_name,
                    "content": f"from {role_name}:  " + msg.content
                })
        else:
            # Add all other messages as is
            converted_messages.append(msg)

    # Now invoke the inventory agent with the properly prepared messages
    result = inventory_agent.invoke({"messages": converted_messages})

    # Take the final message from the SQL agent and append it to the original message history
    inventory_response = result["messages"][-1]

    tagged_response = AIMessage(
        content=inventory_response.content,
        additional_kwargs={**inventory_response.additional_kwargs, "agent_name": "inventory_management_agent"}
    )

    return Command(
        goto="SUPERVISOR_AGENT",
        update={"messages": messages + [tagged_response]}
    )


def sales_analyst_agent_node(state: SupervisorState) -> Command[Literal["SUPERVISOR_AGENT"]]:
    messages = state["messages"]

    system_prompt = SystemMessage(content="""
    You are the Sales Analyst Agent.

    Your role is to provide clear and insightful summaries of recent sales performance based strictly on raw sales data.

    Your workflow:

    1. **Check for existing sales data**:
       - Look through previous messages from the SQL agent and if NO data from SQL agent you MUST to call `get_sales_data_tool`.
       - You must NOT use result/data from Inventory Management agent to make your sales analysis.
       - Data is considered complete only if all records contain valid values for each of the fields:
         `Product_ID`, `Units_Sold`, `Revenue`, and `Date`.
       - When the user requests sales performance for a specific month, you must retrieve data for:
         **(1) the requested month and (2) the immediately preceding month** — this is a strict 2-month window ending in the specified month.
       - Only request or process records that correspond to the **last day of each month** in this period.

    2. **If data is missing or incomplete**:
       - Call `get_sales_data_tool` with a natural-language instruction clearly specifying what is missing.
       - You must justify your tool call, for example: "Missing Revenue field for December 2021" or "No data found for January 2022".
       - Never fabricate or guess data.

    3. **Once complete sales data is available**:
       - Immediately call `summarize_sales_tool` using the raw data and make sure ONLY pass the fields that included in the data from sql_agent, else just pass empty values for those fields that was not included in the raw data.
       - Do NOT summarize, analyze, or manipulate the data yourself — only the tool is allowed to interpret it.

    Rules:
    - NEVER estimate, interpret, or generate synthetic data.
    - NEVER request data more than once unless it's incomplete.
    - ALWAYS use a strict 2-month window: the month requested and the month before it.
    - Focus on delivering factual, data-backed insights only.
    - MUST put your tool call inside <tool_call></tool_call>
    """)

    # Prepare messages
    converted_messages = [system_prompt]
    for msg in messages:
        if isinstance(msg, SystemMessage):
            continue

        elif isinstance(msg, AIMessage):
            role_name = msg.additional_kwargs.get("agent_name", "assistant")

            # Filter out irrelevant SQL messages
            if role_name == "sql_agent":
                intended_for = msg.additional_kwargs.get("intended_for")
                if intended_for != "sales_analyst_agent":
                    continue  # Skip this SQL message if it's not for the sales analyst
            if role_name == "inventory_management_agent":
                continue  #skip messages from inventory management agent

            if msg.content:
                converted_messages.append({
                    "role": "assistant",
                    "name": role_name,
                    "content": f"from {role_name}:  " + msg.content
                })

        else:
            # Human messages or others
            converted_messages.append(msg)

    # Run the sales analyst agent with converted messages
    result = sales_agent.invoke({"messages": converted_messages})

    # Get the latest response
    sales_response = result["messages"][-1]
    tagged_response = AIMessage(
        content=sales_response.content,
        additional_kwargs={**sales_response.additional_kwargs, "agent_name": "sales_analyst_agent"}
    )

    return Command(
        goto="SUPERVISOR_AGENT",
        update={"messages": messages + [tagged_response]}
    )


multi_agent_builder = StateGraph(SupervisorState)
multi_agent_builder.add_node("SUPERVISOR_AGENT", supervisor_node)
multi_agent_builder.add_node("SQL_AGENT", sql_agent_node)
multi_agent_builder.add_node("INVENTORY_MANAGEMENT_AGENT", inventory_management_agent_node)
multi_agent_builder.add_node("SALES_ANALYST_AGENT", sales_analyst_agent_node)

multi_agent_builder.add_edge(START, "SUPERVISOR_AGENT")

multi_agent = multi_agent_builder.compile()

In [ ]:
display(Image(multi_agent.get_graph().draw_mermaid_png()))

In [18]:
messages = [
    # HumanMessage(content="What are the top 15 products sold in store S002 during the first day of January 2022?")
    # HumanMessage(content="What are the total sales for toys category inside store S001 in the month of January 2022?")
    # HumanMessage(content="What should we restock for the store S002 during the end of January 2022?")
    # HumanMessage(content="Can you give me a summary of the sales performance for Store S001 in February 2022?")
    # HumanMessage(content="Retrieve monthly sales data for Store S001 for February 2022 and January 2022")
    # HumanMessage(content="Can you help me create the restock plan for Store S001 at the end of February 2022")
    HumanMessage(
        content="Can you help me create the restock plan for Store S001 at the end of February 2022, and also the sales analysis on the same month?")
]
# Invoke the agent
result = multi_agent.invoke({"messages": messages}, {"recursion_limit": 100}, debug=True)
# Print the result
print("\nFinal conversation:")
for m in result["messages"]:
    print(f"{type(m).__name__} ({m.additional_kwargs.get('agent_name', 'unknown')}): {m.content}")
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f"Tool calls: {m.tool_calls}")

    print("\n")

[-1:checkpoint] State at the end of step -1:
{}
[0:tasks] Starting 1 task for step 0:
- __start__ -> {'messages': [HumanMessage(content='Can you help me create the restock plan for Store S001 at the end of February 2022, and also the sales analysis on the same month?', additional_kwargs={}, response_metadata={})]}
[0:writes] Finished step 0 with writes to 1 channel:
- messages -> [HumanMessage(content='Can you help me create the restock plan for Store S001 at the end of February 2022, and also the sales analysis on the same month?', additional_kwargs={}, response_metadata={})]
[0:checkpoint] State at the end of step 0:
{'messages': [HumanMessage(content='Can you help me create the restock plan for Store S001 at the end of February 2022, and also the sales analysis on the same month?', additional_kwargs={}, response_metadata={})]}
[1:tasks] Starting 1 task for step 1:
- SUPERVISOR_AGENT -> {'messages': [HumanMessage(content='Can you help me create the restock plan for Store S001 at the 